# Практикум: Python и pandas для аналитика

Поэтапный курс с автопроверкой. Каждая задача заканчивается блоком `assert` — если он молчит и печатает галочку, ответ верный. Если падает с `AssertionError`, ответ не сошёлся.

**Как работать**

1. Выполняй ячейки сверху вниз, начиная со следующей — она готовит данные.
2. В задаче пиши свой код вместо строки `...`.
3. Запускай ячейку. Галочка — идёшь дальше.
4. В решение заглядывай только после двух честных попыток. Решение, подсмотренное сразу, не откладывается.

**Правила**

- Не меняй код проверки — он твой судья, а не препятствие.
- Застрял больше чем на 15 минут — открой решение, разбери, закрой и напиши сам по памяти.
- Если ячейка упала с `NameError`, скорее всего ты пропустил ячейку выше.

Данные генерируются прямо здесь, качать ничего не нужно. Это мини-маркетплейс: заказы, товары, продавцы, отзывы — та же схема, что у Olist, только меньше и чище.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(20261001)
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

N_CUST, N_SELL, N_PROD, N_ORD = 400, 40, 120, 2500
CITIES = ['Москва','Санкт-Петербург','Новосибирск','Екатеринбург','Казань','Нижний Новгород','Самара','Омск']
CATS   = ['электроника','мебель','игрушки','красота','спорт','книги','сад','кухня']

customers = pd.DataFrame({
    'customer_id': np.arange(1, N_CUST+1),
    'city': rng.choice(CITIES, N_CUST, p=[.28,.18,.09,.09,.09,.09,.09,.09]),
    'signup_date': pd.to_datetime('2024-01-01') + pd.to_timedelta(rng.integers(0, 300, N_CUST), unit='D'),
})

sellers = pd.DataFrame({
    'seller_id': np.arange(1, N_SELL+1),
    'city': rng.choice(CITIES, N_SELL),
    'rating': np.round(rng.uniform(3.0, 5.0, N_SELL), 1),
})

base = {'электроника':(1500,40000),'мебель':(2000,30000),'игрушки':(300,4000),'красота':(200,5000),
        'спорт':(500,15000),'книги':(200,2000),'сад':(300,9000),'кухня':(400,12000)}
cat = rng.choice(CATS, N_PROD)
products = pd.DataFrame({
    'product_id': np.arange(1, N_PROD+1),
    'category': cat,
    'price': [float(rng.integers(*base[c])) for c in cat],
    'weight_g': rng.integers(50, 12000, N_PROD),
})

cust_for_order = rng.choice(customers.customer_id, N_ORD, p=np.r_[np.repeat(3, 60), np.repeat(1, N_CUST-60)]/ (3*60 + (N_CUST-60)))
purchase = pd.to_datetime('2024-02-01') + pd.to_timedelta(rng.integers(0, 500, N_ORD), unit='D')
status = rng.choice(['delivered','shipped','processing','canceled'], N_ORD, p=[.82,.07,.06,.05])
estimated = purchase + pd.to_timedelta(rng.integers(5, 20, N_ORD), unit='D')
late_days = np.where(rng.random(N_ORD) < 0.21, rng.integers(1, 15, N_ORD), -rng.integers(0, 8, N_ORD))
delivered = estimated + pd.to_timedelta(late_days, unit='D')
delivered = pd.Series(delivered).where(pd.Series(status) == 'delivered')

orders = pd.DataFrame({
    'order_id': np.arange(1, N_ORD+1),
    'customer_id': cust_for_order,
    'status': status,
    'purchase_date': purchase,
    'estimated_date': estimated,
    'delivered_date': delivered,
})

n_items = rng.choice([1,2,3], N_ORD, p=[.7,.22,.08])
rows = []
for oid, k in zip(orders.order_id, n_items):
    for j in range(k):
        pid = int(rng.integers(1, N_PROD+1))
        rows.append((oid, j+1, pid, int(rng.integers(1, N_SELL+1)),
                     float(products.loc[pid-1, 'price']), float(rng.integers(100, 900))))
order_items = pd.DataFrame(rows, columns=['order_id','item_id','product_id','seller_id','price','freight'])

deliv = orders[orders.status == 'delivered']
is_late = (deliv.delivered_date > deliv.estimated_date).values
score = np.where(is_late,
                 rng.choice([1,2,3,4,5], len(deliv), p=[.34,.26,.20,.12,.08]),
                 rng.choice([1,2,3,4,5], len(deliv), p=[.03,.05,.12,.25,.55]))
keep = rng.random(len(deliv)) < 0.85
reviews = pd.DataFrame({
    'review_id': np.arange(1, keep.sum()+1),
    'order_id': deliv.order_id.values[keep],
    'score': score[keep],
    'created_date': deliv.purchase_date.values[keep] + pd.to_timedelta(rng.integers(10, 30, keep.sum()), unit='D'),
})

# немного грязи — как в реальной выгрузке
dirty = rng.choice(order_items.index, 40, replace=False)
order_items.loc[dirty, 'freight'] = np.nan
customers.loc[rng.choice(customers.index, 12, replace=False), 'city'] = None

print('Данные готовы')
for name, d in [('customers',customers),('sellers',sellers),('products',products),
                ('orders',orders),('order_items',order_items),('reviews',reviews)]:
    print(f'{name:13s} {d.shape[0]:>6} строк  {d.shape[1]} колонок')

---
## Блок 1. Чтение таблицы

Прежде чем что-то считать, надо понять, что лежит перед тобой. Эти пять команд аналитик выполняет на автомате при первом взгляде на любые данные.

| команда | что отвечает |
|---|---|
| `df.shape` | сколько строк и колонок |
| `df.info()` | типы колонок и сколько заполнено |
| `df.describe()` | разброс числовых колонок |
| `df.isna().sum()` | где дырки |
| `df.duplicated().sum()` | есть ли полные дубли |

Важная мелочь: `df.head()` печатает таблицу, но ничего не проверяет. Смотреть глазами — не то же самое, что считать.

### Задача 1. Размер таблицы

Положи в `n_rows` число строк в `orders`, в `n_cols` — число колонок.

<details><summary>Подсказка</summary>

`df.shape` отдаёт кортеж из двух чисел.

</details>

<details><summary>Решение</summary>

```python
n_rows, n_cols = orders.shape
```

</details>

In [ ]:
n_rows = ...
n_cols = ...

# --- проверка, не менять ---
assert n_rows == orders.shape[0] and n_cols == orders.shape[1]
print("\u2713 сошлось")

### Задача 2. Где пропуски

Положи в `gaps` объект, где по каждой колонке `order_items` указано число пропусков. Индекс — имена колонок.

<details><summary>Подсказка</summary>

`isna()` даёт таблицу из True/False, а сумма по True — это их количество.

</details>

<details><summary>Решение</summary>

```python
gaps = order_items.isna().sum()
```

</details>

In [ ]:
gaps = ...

# --- проверка, не менять ---
assert gaps['freight'] == 40 and gaps['price'] == 0
print("\u2713 сошлось")

### Задача 3. Уникальные значения

Сколько разных городов у клиентов? Запиши число в `n_cities`. Пропуски городом не считаются.

<details><summary>Подсказка</summary>

`nunique()` пропуски игнорирует по умолчанию.

</details>

<details><summary>Решение</summary>

```python
n_cities = customers.city.nunique()
```

</details>

In [ ]:
n_cities = ...

# --- проверка, не менять ---
assert n_cities == customers.city.nunique()
print("\u2713 сошлось")

### Задача 4. Диапазон дат

В `first_day` и `last_day` положи дату самого раннего и самого позднего заказа.

<details><summary>Решение</summary>

```python
first_day = orders.purchase_date.min()
last_day = orders.purchase_date.max()
```

</details>

In [ ]:
first_day = ...
last_day = ...

# --- проверка, не менять ---
assert first_day == orders.purchase_date.min() and last_day == orders.purchase_date.max()
print("\u2713 сошлось")

### Задача 5. Что внутри категориальной колонки

Положи в `status_counts` количество заказов по каждому статусу, отсортированное по убыванию.

<details><summary>Подсказка</summary>

`value_counts()` сортирует по убыванию сам.

</details>

<details><summary>Решение</summary>

```python
status_counts = orders.status.value_counts()
```

</details>

In [ ]:
status_counts = ...

# --- проверка, не менять ---
assert status_counts.index[0] == 'delivered' and status_counts.sum() == len(orders)
print("\u2713 сошлось")

---
## Блок 2. Фильтрация

Основной рабочий инструмент. Условие внутри скобок — это маска из True/False той же длины, что таблица.

```python
df[df.price > 100]                      # одно условие
df[(df.price > 100) & (df.cat == 'x')]  # и
df[(df.a == 1) | (df.b == 2)]           # или
df[~df.flag]                            # отрицание
df[df.cat.isin(['a','b'])]              # вхождение в список
df[df.city.isna()]                      # пропуски
```

Две вещи, на которых спотыкаются все:

- скобки вокруг каждого условия обязательны, иначе Python сначала применит `>` не к тому;
- пишется `&` и `|`, а не `and` и `or` — последние работают с одним значением, а не с колонкой.

### Задача 6. Дорогие товары

В `expensive` положи строки `products` с ценой выше 20000.

<details><summary>Решение</summary>

```python
expensive = products[products.price > 20000]
```

</details>

In [ ]:
expensive = ...

# --- проверка, не менять ---
assert len(expensive) == (products.price > 20000).sum() and (expensive.price > 20000).all()
print("\u2713 сошлось")

### Задача 7. Два условия

В `big_electro` — электроника дороже 10000.

<details><summary>Подсказка</summary>

Каждое условие в свои скобки, между ними `&`.

</details>

<details><summary>Решение</summary>

```python
big_electro = products[(products.category == 'электроника') & (products.price > 10000)]
```

</details>

In [ ]:
big_electro = ...

# --- проверка, не менять ---
assert len(big_electro) == ((products.category=='электроника') & (products.price>10000)).sum()
print("\u2713 сошлось")

### Задача 8. Отмены и зависшие

В `problem_orders` собери заказы со статусом `canceled` или `processing`. Используй `isin`.

<details><summary>Решение</summary>

```python
problem_orders = orders[orders.status.isin(['canceled','processing'])]
```

</details>

In [ ]:
problem_orders = ...

# --- проверка, не менять ---
assert set(problem_orders.status.unique()) == {'canceled','processing'}
assert len(problem_orders) == orders.status.isin(['canceled','processing']).sum()
print("\u2713 сошлось")

### Задача 9. Клиенты без города

В `no_city` — строки `customers`, где город не заполнен.

<details><summary>Подсказка</summary>

Сравнение с `None` через `==` не работает, нужен `isna()`.

</details>

<details><summary>Решение</summary>

```python
no_city = customers[customers.city.isna()]
```

</details>

In [ ]:
no_city = ...

# --- проверка, не менять ---
assert len(no_city) == customers.city.isna().sum() == 12
print("\u2713 сошлось")

### Задача 10. Фильтр по дате

В `summer` положи заказы, сделанные с 1 июня по 31 августа 2024 включительно.

<details><summary>Подсказка</summary>

Колонку типа datetime можно сравнивать со строкой — pandas сам её разберёт.

</details>

<details><summary>Решение</summary>

```python
summer = orders[(orders.purchase_date >= '2024-06-01') & (orders.purchase_date <= '2024-08-31')]
```

</details>

In [ ]:
summer = ...

# --- проверка, не менять ---
assert len(summer) == ((orders.purchase_date >= '2024-06-01') & (orders.purchase_date <= '2024-08-31')).sum()
print("\u2713 сошлось")

### Задача 11. loc против прямых скобок

Через `.loc` отбери строки `products` категории «книги» и оставь только колонки `product_id` и `price`. Результат в `books`.

<details><summary>Подсказка</summary>

`.loc[строки, колонки]` — две части через запятую.

</details>

<details><summary>Решение</summary>

```python
books = products.loc[products.category == 'книги', ['product_id','price']]
```

</details>

In [ ]:
books = ...

# --- проверка, не менять ---
assert list(books.columns) == ['product_id','price'] and len(books) == (products.category=='книги').sum()
print("\u2713 сошлось")

---
## Блок 3. Новые колонки

Вычисляемые признаки — половина работы аналитика. Три основных способа:

```python
df['total'] = df.price + df.freight          # векторная арифметика, самый быстрый
df['flag']  = np.where(df.x > 10, 'да','нет') # условие в одну строку
df['grp']   = df.cat.map({'a':1,'b':2})       # замена по словарю
```

Про `apply`: он работает построчно и потому медленный. Если то же самое выражается через арифметику или `np.where` — пиши так. `apply` остаётся для случаев, где без питоновской функции не обойтись.

### Задача 12. Сумма позиции

Добавь в `order_items` колонку `total` — цена плюс доставка. Пропуски в доставке считай нулём.

<details><summary>Подсказка</summary>

`fillna(0)` заменяет пропуски до сложения — иначе сумма с NaN даст NaN.

</details>

<details><summary>Решение</summary>

```python
order_items['total'] = order_items.price + order_items.freight.fillna(0)
```

</details>

In [ ]:
...

# --- проверка, не менять ---
assert 'total' in order_items.columns
assert np.isclose(order_items.total.sum(), (order_items.price + order_items.freight.fillna(0)).sum())
print("\u2713 сошлось")

### Задача 13. Флаг просрочки

Добавь в `orders` колонку `is_late`: 1, если доставлено позже обещанного, иначе 0. Для недоставленных тоже 0.

<details><summary>Подсказка</summary>

Сравнение с NaT даёт False — то, что нужно.

</details>

<details><summary>Решение</summary>

```python
orders['is_late'] = (orders.delivered_date > orders.estimated_date).fillna(False).astype(int)
```

</details>

In [ ]:
...

# --- проверка, не менять ---
assert orders.is_late.isin([0,1]).all()
assert orders.is_late.sum() == ((orders.delivered_date > orders.estimated_date).fillna(False)).sum()
print("\u2713 сошлось")

### Задача 14. Срок доставки в днях

Добавь колонку `delivery_days` — сколько дней прошло от покупки до доставки. Целое число дней, у недоставленных NaN.

<details><summary>Подсказка</summary>

Разность дат даёт timedelta, `.dt.days` вытаскивает из неё дни.

</details>

<details><summary>Решение</summary>

```python
orders['delivery_days'] = (orders.delivered_date - orders.purchase_date).dt.days
```

</details>

In [ ]:
...

# --- проверка, не менять ---
assert np.isclose(orders.delivery_days.mean(), (orders.delivered_date - orders.purchase_date).dt.days.mean())
print("\u2713 сошлось")

### Задача 15. Ценовой сегмент

Добавь в `products` колонку `segment`: `низкий` при цене до 1000, `средний` от 1000 до 10000, `высокий` выше. Используй `pd.cut`.

<details><summary>Подсказка</summary>

`bins` задаёт границы, `labels` — названия корзин. Границы включаются справа.

</details>

<details><summary>Решение</summary>

```python
products['segment'] = pd.cut(products.price, bins=[0,1000,10000,np.inf], labels=['низкий','средний','высокий'])
```

</details>

In [ ]:
...

# --- проверка, не менять ---
assert set(products.segment.unique()) <= {'низкий','средний','высокий'}
assert (products.loc[products.price <= 1000,'segment'] == 'низкий').all()
print("\u2713 сошлось")

### Задача 16. Месяц покупки

Добавь в `orders` колонку `month` — месяц покупки строкой вида `2024-06`.

<details><summary>Подсказка</summary>

Аксессор `.dt` открывает всё, что умеют даты.

</details>

<details><summary>Решение</summary>

```python
orders['month'] = orders.purchase_date.dt.strftime('%Y-%m')
```

</details>

In [ ]:
...

# --- проверка, не менять ---
assert (orders.month == orders.purchase_date.dt.strftime('%Y-%m')).all()
assert orders.month.iloc[0].count('-') == 1
print("\u2713 сошлось")

---
## Блок 4. Группировки

`groupby` — ядро аналитики на pandas, прямой аналог `GROUP BY`.

```python
df.groupby('cat').size()                      # число строк в группе
df.groupby('cat').price.mean()                # одна метрика
df.groupby('cat').agg(n=('id','count'),
                      rev=('price','sum'))     # несколько сразу, с именами
df.groupby(['a','b']).price.sum()             # по двум ключам
df.groupby('cat').price.transform('mean')     # среднее группы обратно в строки
```

`agg` против `transform` — любимый вопрос на собеседовании. `agg` схлопывает группу в одно число, `transform` возвращает результат той же длины, что исходная таблица. Второе нужно, когда надо сравнить строку со средним по её группе.

### Задача 17. Заказы по статусам

В `by_status` — количество заказов по каждому статусу.

<details><summary>Решение</summary>

```python
by_status = orders.groupby('status').size()
```

</details>

In [ ]:
by_status = ...

# --- проверка, не менять ---
assert by_status['delivered'] == (orders.status=='delivered').sum()
print("\u2713 сошлось")

### Задача 18. Выручка по категориям

Посчитай суммарную выручку (колонка `price` из `order_items`) по категориям товара. Результат в `cat_rev`, отсортируй по убыванию.

<details><summary>Подсказка</summary>

Категория лежит в products, выручка — в order_items. Сначала соедини.

</details>

<details><summary>Решение</summary>

```python
cat_rev = (order_items.merge(products[['product_id','category']], on='product_id')
             .groupby('category').price.sum().sort_values(ascending=False))
```

</details>

In [ ]:
cat_rev = ...

# --- проверка, не менять ---
ref = order_items.merge(products, on='product_id').groupby('category').price_x.sum().sort_values(ascending=False)
assert np.isclose(cat_rev.iloc[0], ref.iloc[0]) and cat_rev.index[0] == ref.index[0]
print("\u2713 сошлось")

### Задача 19. Несколько метрик сразу

По каждой категории товара посчитай число товаров и среднюю цену. Результат в `cat_stats` с колонками `n` и `avg_price`.

<details><summary>Подсказка</summary>

Именованная агрегация: `имя=(колонка, функция)`.

</details>

<details><summary>Решение</summary>

```python
cat_stats = products.groupby('category').agg(n=('product_id','count'), avg_price=('price','mean'))
```

</details>

In [ ]:
cat_stats = ...

# --- проверка, не менять ---
assert list(cat_stats.columns) == ['n','avg_price']
assert cat_stats.n.sum() == len(products)
print("\u2713 сошлось")

### Задача 20. Средний чек

Посчитай средний чек: сначала сумма `price` по каждому заказу, потом среднее по заказам. Число в `avg_check`.

<details><summary>Подсказка</summary>

Два этажа агрегации. Среднее от price напрямую даст среднюю цену товара — это другая метрика.

</details>

<details><summary>Решение</summary>

```python
avg_check = order_items.groupby('order_id').price.sum().mean()
```

</details>

In [ ]:
avg_check = ...

# --- проверка, не менять ---
assert np.isclose(avg_check, order_items.groupby('order_id').price.sum().mean())
print("\u2713 сошлось")

### Задача 21. Группировка по двум ключам

Посчитай число заказов по паре месяц-статус. Результат в `month_status`.

<details><summary>Подсказка</summary>

Получится Series с двухуровневым индексом.

</details>

<details><summary>Решение</summary>

```python
month_status = orders.groupby(['month','status']).size()
```

</details>

In [ ]:
month_status = ...

# --- проверка, не менять ---
assert month_status.loc[('2024-06','delivered')] == len(orders[(orders.month=='2024-06') & (orders.status=='delivered')])
print("\u2713 сошлось")

### Задача 22. transform: отклонение от среднего по категории

Добавь в `products` колонку `price_vs_cat` — во сколько раз цена товара отличается от средней цены его категории.

<details><summary>Подсказка</summary>

`transform` вернёт колонку той же длины, что таблица — её можно делить напрямую.

</details>

<details><summary>Решение</summary>

```python
products['price_vs_cat'] = products.price / products.groupby('category').price.transform('mean')
```

</details>

In [ ]:
...

# --- проверка, не менять ---
ref = products.price / products.groupby('category').price.transform('mean')
assert np.isclose(products.price_vs_cat.sum(), ref.sum())
print("\u2713 сошлось")

### Задача 23. Топ-N внутри группы

Для каждой категории найди 3 самых дорогих товара. Результат в `top3` — таблица с колонками `category`, `product_id`, `price`.

<details><summary>Подсказка</summary>

Отсортируй всю таблицу, потом возьми `head(3)` внутри каждой группы.

</details>

<details><summary>Решение</summary>

```python
top3 = (products.sort_values('price', ascending=False)
          .groupby('category').head(3)[['category','product_id','price']])
```

</details>

In [ ]:
top3 = ...

# --- проверка, не менять ---
assert len(top3) == products.category.nunique()*3
assert top3.groupby('category').price.max().equals(products.groupby('category').price.max())
print("\u2713 сошлось")

---
## Блок 5. Соединения

```python
a.merge(b, on='id', how='inner')              # только совпавшие
a.merge(b, on='id', how='left')               # все строки слева
a.merge(b, left_on='x', right_on='y')         # разные имена ключей
a.merge(b, on='id', validate='many_to_one')   # проверка кратности
pd.concat([a, b])                              # склейка сверху вниз
```

Главная опасность джойна — размножение строк. Если в правой таблице ключ не уникален, каждая строка слева продублируется, и любая сумма после этого станет завышенной. `validate='many_to_one'` падает с ошибкой, когда это происходит. Привычка его ставить спасает от целого класса молчаливых ошибок в отчётах.

### Задача 24. Простое соединение

Соедини `order_items` с `products` по `product_id`, оставь колонки `order_id`, `product_id`, `category`, `price`. Результат в `items_cat`.

<details><summary>Подсказка</summary>

Бери из products только нужные колонки — иначе получишь price_x и price_y.

</details>

<details><summary>Решение</summary>

```python
items_cat = (order_items.merge(products[['product_id','category']], on='product_id')
               [['order_id','product_id','category','price']])
```

</details>

In [ ]:
items_cat = ...

# --- проверка, не менять ---
assert len(items_cat) == len(order_items)
assert set(items_cat.columns) == {'order_id','product_id','category','price'}
print("\u2713 сошлось")

### Задача 25. Проверка кратности

Повтори прошлое соединение, добавив `validate='many_to_one'`. Результат в `items_safe`. Если упадёт — значит ключ не уникален справа.

<details><summary>Решение</summary>

```python
items_safe = order_items.merge(products[['product_id','category']], on='product_id', validate='many_to_one')
```

</details>

In [ ]:
items_safe = ...

# --- проверка, не менять ---
assert len(items_safe) == len(order_items)
print("\u2713 сошлось")

### Задача 26. Левое соединение и поиск пропавших

Найди товары, которых нет ни в одном заказе. Результат в `never_sold` — таблица из products.

<details><summary>Подсказка</summary>

Либо через `isin` с отрицанием — так короче и тоже правильно.

</details>

<details><summary>Решение</summary>

```python
m = products.merge(order_items[['product_id']].drop_duplicates(), on='product_id', how='left', indicator=True)
never_sold = products[m._merge.values == 'left_only']
```

</details>

In [ ]:
never_sold = ...

# --- проверка, не менять ---
ref = set(products.product_id) - set(order_items.product_id)
assert set(never_sold.product_id) == ref
print("\u2713 сошлось")

### Задача 27. Соединение трёх таблиц

Собери таблицу `full`: позиции заказа + категория товара + город клиента. Колонки: `order_id`, `category`, `city`, `price`.

<details><summary>Подсказка</summary>

Merge можно ставить цепочкой, по одному соединению за раз.

</details>

<details><summary>Решение</summary>

```python
full = (order_items.merge(products[['product_id','category']], on='product_id')
          .merge(orders[['order_id','customer_id']], on='order_id')
          .merge(customers[['customer_id','city']], on='customer_id')
          [['order_id','category','city','price']])
```

</details>

In [ ]:
full = ...

# --- проверка, не менять ---
assert set(full.columns) == {'order_id','category','city','price'}
assert len(full) == len(order_items)
print("\u2713 сошлось")

### Задача 28. Выручка по городам

Посчитай выручку по городам клиентов. Результат в `city_rev`, по убыванию.

<details><summary>Решение</summary>

```python
city_rev = (order_items.merge(orders[['order_id','customer_id']], on='order_id')
              .merge(customers[['customer_id','city']], on='customer_id')
              .groupby('city').price.sum().sort_values(ascending=False))
```

</details>

In [ ]:
city_rev = ...

# --- проверка, не менять ---
ref = (order_items.merge(orders[['order_id','customer_id']], on='order_id')
         .merge(customers[['customer_id','city']], on='customer_id')
         .groupby('city').price.sum().sort_values(ascending=False))
assert city_rev.index[0] == ref.index[0] and np.isclose(city_rev.iloc[0], ref.iloc[0])
print("\u2713 сошлось")

---
## Блок 6. Время

```python
df.purchase_date.dt.year / .month / .day / .dayofweek
df.dt.strftime('%Y-%m')                     # в строку нужного формата
df.dt.to_period('M')                        # период-месяц
df.set_index('date').resample('ME').sum()   # агрегация по времени
df.shift(1)                                 # сдвиг на строку — аналог LAG
df.cumsum()                                 # накопительный итог
```

Прирост к прошлому периоду считается через `shift`, накопленный итог — через `cumsum`. Это прямые аналоги оконных функций `LAG` и `SUM OVER (ORDER BY ...)` из SQL.

### Задача 29. Выручка по месяцам

Посчитай выручку по месяцам покупки. Результат в `monthly` — Series, индекс строкой `YYYY-MM`, по возрастанию.

<details><summary>Решение</summary>

```python
monthly = (order_items.merge(orders[['order_id','month']], on='order_id')
             .groupby('month').price.sum().sort_index())
```

</details>

In [ ]:
monthly = ...

# --- проверка, не менять ---
ref = order_items.merge(orders[['order_id','month']], on='order_id').groupby('month').price.sum().sort_index()
assert np.isclose(monthly.sum(), ref.sum()) and list(monthly.index) == list(ref.index)
print("\u2713 сошлось")

### Задача 30. Прирост месяц к месяцу

На основе `monthly` посчитай прирост в процентах к прошлому месяцу. Результат в `growth`, первое значение — NaN.

<details><summary>Подсказка</summary>

`pct_change()` делает ровно это. Вручную было бы `(m - m.shift(1)) / m.shift(1)`.

</details>

<details><summary>Решение</summary>

```python
growth = monthly.pct_change() * 100
```

</details>

In [ ]:
growth = ...

# --- проверка, не менять ---
ref = monthly.pct_change()*100
assert np.isclose(growth.dropna().sum(), ref.dropna().sum())
print("\u2713 сошлось")

### Задача 31. Накопленная выручка

Накопительный итог по `monthly`. Результат в `cum_rev`, последнее значение должно совпасть с общей суммой.

<details><summary>Решение</summary>

```python
cum_rev = monthly.cumsum()
```

</details>

In [ ]:
cum_rev = ...

# --- проверка, не менять ---
assert np.isclose(cum_rev.iloc[-1], monthly.sum())
print("\u2713 сошлось")

### Задача 32. День недели

В какой день недели делают больше всего заказов? Номер дня (0 — понедельник) в `busiest_day`.

<details><summary>Подсказка</summary>

`idxmax()` отдаёт индекс максимального значения, а не само значение.

</details>

<details><summary>Решение</summary>

```python
busiest_day = orders.purchase_date.dt.dayofweek.value_counts().idxmax()
```

</details>

In [ ]:
busiest_day = ...

# --- проверка, не менять ---
assert busiest_day == orders.purchase_date.dt.dayofweek.value_counts().idxmax()
print("\u2713 сошлось")

### Задача 33. Скользящее среднее

Сгладь `monthly` скользящим средним по 3 месяцам. Результат в `smooth`.

<details><summary>Подсказка</summary>

`rolling(n)` открывает окно из n строк, дальше любая агрегация.

</details>

<details><summary>Решение</summary>

```python
smooth = monthly.rolling(3).mean()
```

</details>

In [ ]:
smooth = ...

# --- проверка, не менять ---
assert np.isclose(smooth.dropna().iloc[0], monthly.iloc[:3].mean())
print("\u2713 сошлось")

---
## Блок 7. Сводные таблицы и когорты

`pivot_table` разворачивает длинную таблицу в широкую: одно поле в строки, другое в колонки.

```python
df.pivot_table(index='a', columns='b', values='x', aggfunc='sum', fill_value=0)
```

Когортный анализ — главная вещь этого блока и обязательный пункт портфолио. Логика в три шага: за каждым клиентом закрепляем месяц первой покупки, считаем, сколько месяцев прошло от неё до каждой следующей, разворачиваем в таблицу когорта × возраст.

### Задача 34. Сводная категория × месяц

Построй сводную: строки — категория, колонки — месяц, значения — сумма `price`, пропуски нулями. Результат в `pt`.

<details><summary>Решение</summary>

```python
base = (order_items.merge(products[['product_id','category']], on='product_id')
          .merge(orders[['order_id','month']], on='order_id'))
pt = base.pivot_table(index='category', columns='month', values='price', aggfunc='sum', fill_value=0)
```

</details>

In [ ]:
pt = ...

# --- проверка, не менять ---
base = order_items.merge(products[['product_id','category']], on='product_id').merge(orders[['order_id','month']], on='order_id')
ref = base.pivot_table(index='category', columns='month', values='price', aggfunc='sum', fill_value=0)
assert pt.shape == ref.shape and np.isclose(pt.values.sum(), ref.values.sum())
print("\u2713 сошлось")

### Задача 35. Месяц первой покупки

Для каждого клиента определи месяц его первого заказа. Результат в `first_month` — Series, индекс `customer_id`, значения строкой `YYYY-MM`.

<details><summary>Решение</summary>

```python
first_month = orders.groupby('customer_id').purchase_date.min().dt.strftime('%Y-%m')
```

</details>

In [ ]:
first_month = ...

# --- проверка, не менять ---
ref = orders.groupby('customer_id').purchase_date.min().dt.strftime('%Y-%m')
assert first_month.equals(ref)
print("\u2713 сошлось")

### Задача 36. Возраст когорты

Добавь в `orders` колонку `cohort` (месяц первой покупки клиента) и `period` — сколько полных месяцев прошло от первой покупки до этого заказа.

<details><summary>Подсказка</summary>

Разницу в месяцах удобно считать как `(год1-год2)*12 + (месяц1-месяц2)`.

</details>

<details><summary>Решение</summary>

```python
orders['cohort'] = orders.customer_id.map(first_month)
fm = pd.to_datetime(orders.cohort + '-01')
mm = pd.to_datetime(orders.month + '-01')
orders['period'] = (mm.dt.year - fm.dt.year) * 12 + (mm.dt.month - fm.dt.month)
```

</details>

In [ ]:
...

# --- проверка, не менять ---
assert orders.period.min() == 0
assert (orders.groupby('customer_id').period.min() == 0).all()
print("\u2713 сошлось")

### Задача 37. Когортная таблица

Построй `cohort_table`: строки — когорта, колонки — `period`, значения — число уникальных клиентов.

<details><summary>Решение</summary>

```python
cohort_table = orders.pivot_table(index='cohort', columns='period', values='customer_id', aggfunc='nunique')
```

</details>

In [ ]:
cohort_table = ...

# --- проверка, не менять ---
assert cohort_table.shape[0] == orders.cohort.nunique()
assert cohort_table[0].sum() == orders.customer_id.nunique()
print("\u2713 сошлось")

### Задача 38. Retention в процентах

Переведи `cohort_table` в доли от размера когорты. Результат в `retention` — в колонке 0 везде единицы.

<details><summary>Подсказка</summary>

`divide(..., axis=0)` делит каждую строку на своё число.

</details>

<details><summary>Решение</summary>

```python
retention = cohort_table.divide(cohort_table[0], axis=0)
```

</details>

In [ ]:
retention = ...

# --- проверка, не менять ---
assert np.allclose(retention[0].dropna().values, 1.0)
print("\u2713 сошлось")

---
## Блок 8. Метрики маркетплейса

Последний блок — то, ради чего всё предыдущее. Здесь нет новых функций, только бизнес-вопросы. Ровно такие дают в тестовом задании.

### Задача 39. Конверсия в доставку

Какая доля всех заказов дошла до статуса delivered? Число от 0 до 1 в `conversion`.

<details><summary>Подсказка</summary>

Среднее по колонке из True/False и есть доля.

</details>

<details><summary>Решение</summary>

```python
conversion = (orders.status == 'delivered').mean()
```

</details>

In [ ]:
conversion = ...

# --- проверка, не менять ---
assert np.isclose(conversion, (orders.status=='delivered').mean())
print("\u2713 сошлось")

### Задача 40. Доля повторных клиентов

Какая доля клиентов сделала больше одного заказа? Число в `repeat_share`.

<details><summary>Решение</summary>

```python
repeat_share = (orders.groupby('customer_id').size() > 1).mean()
```

</details>

In [ ]:
repeat_share = ...

# --- проверка, не менять ---
assert np.isclose(repeat_share, (orders.groupby('customer_id').size() > 1).mean())
print("\u2713 сошлось")

### Задача 41. Цена просрочки

Сравни средний балл отзыва у заказов, доставленных вовремя и с опозданием. Результат в `score_by_late` — Series с индексом 0 и 1.

<details><summary>Решение</summary>

```python
score_by_late = (reviews.merge(orders[['order_id','is_late']], on='order_id')
                   .groupby('is_late').score.mean())
```

</details>

In [ ]:
score_by_late = ...

# --- проверка, не менять ---
ref = reviews.merge(orders[['order_id','is_late']], on='order_id').groupby('is_late').score.mean()
assert np.isclose(score_by_late.loc[1], ref.loc[1])
print("\u2713 сошлось")

### Задача 42. Проблемные продавцы

Найди продавцов с минимум 30 доставленными заказами и долей просрочек выше 25%. Результат в `bad_sellers` — таблица с колонками `n` и `late_share`, по убыванию `late_share`.

<details><summary>Подсказка</summary>

Сначала убери дубли пары продавец-заказ, иначе заказ из трёх позиций посчитается трижды.

</details>

<details><summary>Решение</summary>

```python
base = order_items.merge(orders[['order_id','status','is_late']], on='order_id')
base = base[base.status=='delivered'].drop_duplicates(['seller_id','order_id'])
g = base.groupby('seller_id').agg(n=('order_id','nunique'), late_share=('is_late','mean'))
bad_sellers = g[(g.n >= 30) & (g.late_share > 0.25)].sort_values('late_share', ascending=False)
```

</details>

In [ ]:
bad_sellers = ...

# --- проверка, не менять ---
base = order_items.merge(orders[['order_id','status','is_late']], on='order_id')
base = base[base.status=='delivered'].drop_duplicates(['seller_id','order_id'])
g = base.groupby('seller_id').agg(n=('order_id','nunique'), late_share=('is_late','mean'))
ref = g[(g.n>=30)&(g.late_share>0.25)]
assert len(bad_sellers) == len(ref)
print("\u2713 сошлось")

### Задача 43. ABC-анализ

Отсортируй категории по выручке и посчитай накопленную долю. Результат в `abc` — таблица с колонками `revenue` и `cum_share`.

<details><summary>Решение</summary>

```python
abc = (order_items.merge(products[['product_id','category']], on='product_id')
         .groupby('category').price.sum().sort_values(ascending=False).to_frame('revenue'))
abc['cum_share'] = abc.revenue.cumsum() / abc.revenue.sum()
```

</details>

In [ ]:
abc = ...

# --- проверка, не менять ---
assert np.isclose(abc.cum_share.iloc[-1], 1.0)
assert abc.revenue.is_monotonic_decreasing
print("\u2713 сошлось")

### Задача 44. Средний чек по сегментам

Сравни средний чек у повторных клиентов и у тех, кто заказал один раз. Результат в `check_by_type` — Series с индексом False и True (True — повторные).

<details><summary>Решение</summary>

```python
ord_tot = order_items.groupby('order_id').price.sum().rename('total')
b = orders[['order_id','customer_id']].merge(ord_tot, on='order_id')
b['repeat'] = b.customer_id.map(orders.groupby('customer_id').size() > 1)
check_by_type = b.groupby('repeat').total.mean()
```

</details>

In [ ]:
check_by_type = ...

# --- проверка, не менять ---
ord_tot = order_items.groupby('order_id').price.sum().rename('total')
b = orders[['order_id','customer_id']].merge(ord_tot, on='order_id')
b['repeat'] = b.customer_id.map(orders.groupby('customer_id').size() > 1)
ref = b.groupby('repeat').total.mean()
assert np.isclose(check_by_type.loc[True], ref.loc[True])
print("\u2713 сошлось")

### Задача 45. Среднее против медианы

Посчитай среднее и медиану суммы заказа. Положи в `mean_check` и `median_check`, а в `skew_comment` — строку `правый хвост`, если среднее заметно больше медианы, иначе `симметрично`.

<details><summary>Подсказка</summary>

Если среднее сильно выше медианы, распределение тянет вправо, и среднее как метрика вводит в заблуждение.

</details>

<details><summary>Решение</summary>

```python
t = order_items.groupby('order_id').price.sum()
mean_check, median_check = t.mean(), t.median()
skew_comment = 'правый хвост' if mean_check > median_check * 1.1 else 'симметрично'
```

</details>

In [ ]:
mean_check = ...
median_check = ...
skew_comment = ...

# --- проверка, не менять ---
t = order_items.groupby('order_id').price.sum()
assert np.isclose(mean_check, t.mean()) and np.isclose(median_check, t.median())
assert skew_comment == ('правый хвост' if t.mean() > t.median()*1.1 else 'симметрично')
print("\u2713 сошлось")

---
## Дальше

Если дошёл сюда и все галочки на месте — базы pandas тебе хватает для стажёрского собеседования. Что делать с этим:

1. Открой `stats_practice.ipynb` — там то же самое по статистике и A/B.
2. Повтори блок 8 на настоящем Olist. Те же вопросы, но данные грязные и объём больше — это и есть разница между учебной задачей и работой.
3. Любую задачу из блоков 4-8 попробуй написать ещё и в SQL. Если цифры сошлись, значит понял обе версии, а не заучил синтаксис.

Застрявшие места не копи молча — приноси, разберём.